Verify Tables

In [0]:
%%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,customer_table,false
default,dim_customer,false
default,dim_date,false
default,dim_product,false
default,dim_store,false
default,fact_sales,false


DataFrame[database: string, tableName: string, isTemporary: boolean]

Create Business Views

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_sales_summary AS
SELECT
    ds.state,
    dp.category,
    COUNT(fs.sale_id) AS total_orders,
    SUM(fs.quantity) AS total_quantity,
    ROUND(SUM(fs.amount),2) AS total_sales
FROM workspace.default.fact_sales fs
JOIN workspace.default.dim_store ds
ON fs.store_id = ds.store_id
JOIN workspace.default.dim_product dp
ON fs.product_id = dp.product_id
GROUP BY ds.state, dp.category;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_monthly_sales AS
SELECT
    YEAR(sale_date) AS year,
    MONTH(sale_date) AS month,
    ROUND(SUM(amount),2) AS total_sales
FROM workspace.default.fact_sales
GROUP BY YEAR(sale_date), MONTH(sale_date)
ORDER BY year, month;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_top_products AS
SELECT
    dp.product_name,
    dp.category,
    ROUND(SUM(fs.amount),2) AS revenue
FROM workspace.default.fact_sales fs
JOIN workspace.default.dim_product dp
ON fs.product_id = dp.product_id
GROUP BY dp.product_name, dp.category
ORDER BY revenue DESC;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_customer_sales AS
SELECT
    dc.customer_name,
    dc.city,
    ROUND(SUM(fs.amount),2) AS total_spent,
    COUNT(fs.sale_id) AS total_orders
FROM workspace.default.fact_sales fs
JOIN workspace.default.dim_customer dc
ON fs.customer_id = dc.customer_id
GROUP BY dc.customer_name, dc.city
ORDER BY total_spent DESC;

Business Queries

In [0]:
%sql
SELECT ROUND(SUM(amount),2) AS total_revenue
FROM workspace.default.fact_sales;

total_revenue
1.420878105E7


In [0]:
%sql
SELECT
    ds.state,
    ROUND(SUM(fs.amount),2) AS revenue
FROM workspace.default.fact_sales fs
JOIN workspace.default.dim_store ds
ON fs.store_id = ds.store_id
GROUP BY ds.state
ORDER BY revenue DESC;

state,revenue
Maharashtra,4525927.08
Uttar Pradesh,2217779.74
West Bengal,2054720.56
Karnataka,1608395.28
Tamil Nadu,1559102.71
Telangana,1140680.62
Delhi,1102175.06


In [0]:
%sql
SELECT
    dp.product_name,
    ROUND(SUM(fs.amount),2) AS revenue
FROM workspace.default.fact_sales fs
JOIN workspace.default.dim_product dp
ON fs.product_id = dp.product_id
GROUP BY dp.product_name
ORDER BY revenue DESC
LIMIT 10;

product_name,revenue
Mollitia,286228.16
Sed,214101.7
Nam,210920.04
Saepe,210726.59
Dignissimos,202631.24
Sint,192070.68
Sunt,172843.87
Ipsa,171289.63
Nostrum,164212.02
Facilis,160420.69


In [0]:
%sql
SELECT
    YEAR(sale_date) AS year,
    MONTH(sale_date) AS month,
    ROUND(SUM(amount),2) AS revenue
FROM workspace.default.fact_sales
GROUP BY YEAR(sale_date), MONTH(sale_date)
ORDER BY year, month;

year,month,revenue
2024,7,480883.3
2024,8,601789.96
2024,9,579766.32
2024,10,617554.3
2024,11,648129.34
2024,12,599906.12
2025,1,614512.27
2025,2,552917.31
2025,3,575901.52
2025,4,588270.34


Verify Views

In [0]:
%sql
SELECT * FROM workspace.default.vw_sales_summary LIMIT 10;

SELECT * FROM workspace.default.vw_monthly_sales LIMIT 10;

SELECT * FROM workspace.default.vw_top_products LIMIT 10;

SELECT * FROM workspace.default.vw_customer_sales LIMIT 10;

customer_name,city,total_spent,total_orders
Dayita Dhillon,Lucknow,35152.09,18
Pushti Ratta,Mumbai,34470.71,17
Raagini Savant,Pune,33863.15,16
Harish Sood,Kolkata,32079.9,16
Bachittar Menon,Mumbai,31038.44,13
Ganga Tiwari,Pune,30688.52,17
Aarini Lata,Bangalore,30093.5,15
Ekani Ramaswamy,Bangalore,29699.73,19
Udyati Gara,Chennai,29696.18,15
Amaira Mukhopadhyay,Hyderabad,28885.22,22
